# FAISS Vector Search and Conversational RAG

This notebook demonstrates a FAISS vector-store workflow: prepare and split documents, embed and index chunks, persist or reload the index, retrieve relevant context, and use it in a conversational RAG chain. FAISS supports efficient local similarity search; the chain illustrates how retrieval grounds LLM answers and how history supports follow-up questions.

In [46]:
# import liabraries
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

### Data ingesion and processing


In [47]:
documents = [
    Document(
        page_content="Machine Learning (ML) is a branch of Artificial Intelligence that enables systems to learn patterns from data and make predictions without being explicitly programmed. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.",
        metadata={
            "topic": "Machine Learning",
            "category": "AI",
            "source": "ml_guide.txt"
        }
    ),
    Document(
        page_content="Supervised learning trains a model using labeled data. Common algorithms include Linear Regression for predicting continuous values, Logistic Regression for classification, and Decision Trees for making predictions based on feature conditions.",
        metadata={
            "topic": "Supervised Learning",
            "category": "Machine Learning",
            "source": "supervised_learning.txt"
        }
    ),
    Document(
        page_content="Retrieval-Augmented Generation (RAG) combines information retrieval with Large Language Models (LLMs). It retrieves relevant information from external documents and provides that context to the LLM to generate grounded, context-aware answers.",
        metadata={
            "topic": "RAG",
            "category": "Generative AI",
            "source": "rag_basics.txt"
        }
    ),
    Document(
        page_content="A typical RAG pipeline includes document loading, text splitting, embedding generation, vector database storage, similarity search, context retrieval, and LLM response generation. Chunk size, chunk overlap, embedding quality, and retrieval strategy affect answer quality.",
        metadata={
            "topic": "RAG Pipeline",
            "category": "Generative AI",
            "source": "rag_pipeline.txt"
        }
    ),
    Document(
        page_content="LangChain is a framework for building applications powered by Large Language Models. It provides integrations for document loaders, text splitters, embedding models, vector stores, retrievers, prompts, and language models, helping developers compose these components into RAG workflows.",
        metadata={
            "topic": "LangChain",
            "category": "LLM Framework",
            "source": "langchain_basics.txt"
        }
    )
]

### Chunks creation | Splitting documents

In [48]:
split_docs = RecursiveCharacterTextSplitter(
    separators=["\n\n", " ", "."],
    chunk_size=100,
    chunk_overlap=25,
)

chunks = split_docs.split_documents(documents)
print(f"Chunk Size: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk of {i+1}: {chunk.page_content}")
    

Chunk Size: 19
Chunk of 1: Machine Learning (ML) is a branch of Artificial Intelligence that enables systems to learn patterns
Chunk of 2: to learn patterns from data and make predictions without being explicitly programmed. Common
Chunk of 3: programmed. Common approaches include supervised learning, unsupervised learning, and reinforcement
Chunk of 4: and reinforcement learning.
Chunk of 5: Supervised learning trains a model using labeled data. Common algorithms include Linear Regression
Chunk of 6: Linear Regression for predicting continuous values, Logistic Regression for classification, and
Chunk of 7: for classification, and Decision Trees for making predictions based on feature conditions.
Chunk of 8: Retrieval-Augmented Generation (RAG) combines information retrieval with Large Language Models
Chunk of 9: Large Language Models (LLMs). It retrieves relevant information from external documents and
Chunk of 10: external documents and provides that context to the LLM to generate gr

### Embed the chunks into vectors

In [49]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [50]:
embed = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

In [51]:
text = ["Hey I am here to searchi for you", "AI"]
ventors = await embed.aembed_documents(text)
ventors[0]


[0.0198211669921875,
 -0.04510498046875,
 -0.0183868408203125,
 0.004230499267578125,
 -0.04437255859375,
 -0.005016326904296875,
 -0.0216217041015625,
 0.0236358642578125,
 -0.0271759033203125,
 -0.029815673828125,
 -0.005130767822265625,
 0.0113372802734375,
 0.0290374755859375,
 -0.02764892578125,
 0.0364990234375,
 0.0389404296875,
 -0.05023193359375,
 0.0252227783203125,
 0.0033416748046875,
 0.0224151611328125,
 0.0247344970703125,
 -0.01090240478515625,
 -0.05084228515625,
 0.023712158203125,
 0.046295166015625,
 -0.006694793701171875,
 0.0295257568359375,
 0.050628662109375,
 0.0192413330078125,
 0.0291748046875,
 0.0164337158203125,
 -0.037017822265625,
 -0.03472900390625,
 0.01049041748046875,
 0.00441741943359375,
 -0.01285552978515625,
 -0.0001302957534790039,
 -0.004413604736328125,
 -0.0157470703125,
 -0.04498291015625,
 -0.004985809326171875,
 -0.012939453125,
 0.0706787109375,
 -0.00836181640625,
 -0.00986480712890625,
 -0.014251708984375,
 -0.01345062255859375,
 -0.019

In [52]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embed
)
vectorstore

In [53]:
vectorstore.save_local("faiss")

In [54]:
loaded = vectorstore.load_local("faiss", embed, allow_dangerous_deserialization=True)
loaded.index.ntotal

19

In [55]:
#Similarity Search

# query = "Hi, Please tell me aboout AI"
# vectorstore_result = vectorstore.similarity_search(query, k=3)
# vectorstore_result

In [56]:
query = "Hi, Please tell me aboout ML"
vectorstore_result = vectorstore.similarity_search_with_score(query, k=3)
for doc, score in vectorstore_result:
    print(f"Score: {score: .3f}")
    print(f"Docs of metadata: {doc.metadata}")
    print(f"Docs: {doc.page_content}")

Score:  0.849
Docs of metadata: {'topic': 'Machine Learning', 'category': 'AI', 'source': 'ml_guide.txt'}
Docs: Machine Learning (ML) is a branch of Artificial Intelligence that enables systems to learn patterns
Score:  1.293
Docs of metadata: {'topic': 'LangChain', 'category': 'LLM Framework', 'source': 'langchain_basics.txt'}
Docs: Models. It provides integrations for document loaders, text splitters, embedding models, vector
Score:  1.315
Docs of metadata: {'topic': 'RAG', 'category': 'Generative AI', 'source': 'rag_basics.txt'}
Docs: Large Language Models (LLMs). It retrieves relevant information from external documents and


In [57]:
#Now - Call Model and create retriever
llm = ChatOpenAI(
    model="gpt-4o-mini"
)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x12742d590>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12742e490>, root_client=<openai.OpenAI object at 0x124b9e780>, root_async_client=

In [58]:
answer_prompt = ChatPromptTemplate.from_template(
"""Anser the question based on given context, If you do not know then say , I do not have context!
   Context: {context}
   Question: {question}
   Answer:
"""
)

answer_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Anser the question based on given context, If you do not know then say , I do not have context!\n   Context: {context}\n   Question: {question}\n   Answer:\n'), additional_kwargs={})])

In [59]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    kwargs={"k":3}
)

retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x132768640>, search_kwargs={})

In [60]:

from typing import List
def format_docs(docs: List[Document]) -> str:
    ### Formating docs in simple string format"""
    formatted=[]
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'unknown')
        formatted.append(f"Document {i+1} (Source: {source}): \n{doc.page_content}")

    return "\n\n".join(formatted)

In [63]:
from langchain_core.output_parsers import StrOutputParser
simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | answer_prompt
    | llm
    | StrOutputParser()
)
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x132768640>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Anser the question based on given context, If you do not know then say , I do not have context!\n   Context: {context}\n   Question: {question}\n   Answer:\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_

In [64]:
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Please rewrite/update question based on given context, do not answer"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

history_awa_retriever = create_history_aware_retriever(llm, retriever, rewrite_prompt)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """Please answer based on given context and details, if you not know, say i dont know \n Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}")
    
])

answer_chain = create_stuff_documents_chain(llm, answer_prompt)

rag_conv_chain = create_retrieval_chain(history_awa_retriever, answer_chain)

rag_conv_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x132768640>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[lang

In [69]:
chat_history = []
def ask(question: str):
    result = rag_conv_chain.invoke(
        {"input": question, "chat_history": chat_history}
    )
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=result["answer"])
    ])
    return result

In [70]:
multiple_question = [
    "What is Machine learning",
    "For what ML used",
    "What is AI"
]

for i, que in enumerate(multiple_question):
    result = ask(que)
    print(f"Anser of {i+1} question: {result["answer"]}")

Anser of 1 question: Machine Learning (ML) is a branch of Artificial Intelligence that enables systems to learn patterns from data and make predictions without being explicitly programmed. It involves using algorithms and statistical models to analyze and interpret complex data sets. Common approaches in machine learning include supervised learning, unsupervised learning, and reinforcement learning.
Anser of 2 question: Machine Learning (ML) is used for a wide range of applications, including but not limited to:

1. **Predictive Analytics**: Forecasting future trends based on historical data, such as sales forecasting and risk assessment.

2. **Image Recognition**: Analyzing and classifying images for applications like facial recognition, medical imaging, and autonomous vehicles.

3. **Natural Language Processing (NLP)**: Understanding and generating human language, which is used in applications like chatbots, translation services, and sentiment analysis.

4. **Recommendation Systems**